# Fine-tune the dense encoder on NFCorpus (Kaggle, GPU)

Kaggle version of `finetune_embedder_colab.ipynb`. It fine-tunes `BAAI/bge-small-en-v1.5` with
`MultipleNegativesRankingLoss` (in-batch negatives, no duplicate queries per batch). All logic lives in
the package (`evidence_ir.finetune.run_finetune_experiment`, tested on CPU with a fake encoder), and this
notebook only calls it.

Procedure:

1. Train on the NFCorpus **train** qrels only.
2. After every epoch (up to 3), score dense nDCG@10 on the **dev** split with the package's metric code.
   A small learning-rate grid (1e-5, 2e-5, 5e-5) is searched the same way. The untuned base model is a
   dev candidate too, so if fine-tuning never helps on dev the base model is kept.
3. Keep only the best dev checkpoint.
4. Evaluate the base and the selected encoder **once** on **test** (BM25, dense, hybrid RRF) and compare
   them with the package's paired bootstrap.


**Setup (once):** a Kaggle account with a verified phone number. In the notebook's right-hand panel, set
**Accelerator: GPU T4 x2**, **Internet: on**, and **Environment: Always use latest environment**.

**Run:** click **Save Version > Save & Run All (Commit)**. The run continues with the browser closed.
When it finishes, open the version's **Output** tab and download `finetune_results.zip`
(`finetune_metrics.json`, `finetune_report.md`, `finetune_per_query_test.csv`).

In [ ]:
# Use one GPU: the trainer would otherwise try to spread the model across both T4s.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Colab ships torchao 0.10, which breaks recent peft/transformers imports; it is not needed here.
!pip uninstall -y -q torchao
# Force-reinstall so a reused runtime never keeps an older copy of the package.
!pip install -q --force-reinstall --no-deps git+https://github.com/rkesseku/clinical-evidence-retrieval.git
!pip install -q "sentence-transformers>=3.0" "datasets>=2.20" fastapi httpx

In [ ]:
# Plain data files from the official BEIR mirror (no Hub loading scripts).
!wget -q -nc https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip
!unzip -q -o nfcorpus.zip
!ls nfcorpus nfcorpus/qrels

In [ ]:
import torch
from evidence_ir.finetune import (SentenceTransformerBackend, build_training_pairs, check_disjoint,
                                  estimate_minutes, load_finetune_data, run_finetune_experiment)

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("GPU:", GPU)
LRS, MAX_EPOCHS, BATCH = (1e-5, 2e-5, 5e-5), 3, 64
OUT = "finetune_results"

data = load_finetune_data("nfcorpus")
check_disjoint(data.train_qrels, data.dev, data.test)  # train / dev / test queries are disjoint
n_pairs = len(build_training_pairs(data.train_qrels, data.queries, data.corpus))
print(f"train: {len(data.train_qrels):,} queries, {n_pairs:,} pairs | dev: {len(data.dev.queries)} "
      f"| test: {len(data.test.queries)} | corpus: {len(data.corpus):,}")
est = estimate_minutes(n_pairs, BATCH, MAX_EPOCHS, len(LRS))
print(f"estimated time on a T4 (assumed throughput, not measured): ~{est['total_minutes']:.0f} min "
      f"({est['train_steps']:,} training steps, {est['train_minutes']:.0f} min training, "
      f"{est['eval_minutes']:.0f} min evaluation)")

In [ ]:
backend = SentenceTransformerBackend("BAAI/bge-small-en-v1.5", device="cuda", batch_size=BATCH,
                                     seed=13, work_dir="ft_work")
result = run_finetune_experiment(
    data, backend, OUT, lrs=LRS, max_epochs=MAX_EPOCHS, n_boot=2000, seed=0,
    extra_info={"gpu": GPU, "hyperparameters": backend.hyperparameters()},
)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/finetune_report.md", encoding="utf-8").read()))

In [ ]:
import shutil

# The checkpoint (~130 MB) is left out of the zip: only metrics, report and per-query scores.
shutil.rmtree(f"{OUT}/best_model", ignore_errors=True)
shutil.make_archive("/kaggle/working/finetune_results", "zip", OUT)
print("Download finetune_results.zip from this version's Output tab.")